# Section 7: Split Design and Grouped Leakage Control
Develops strict Stratified Group partitioning bounding leakages without rescanning source images.

In [7]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

OUTPUT_ROOT   = r"C:\SKIN CANCER v2\pipe output"
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_quality.csv")

# Fall back to post-metadata manifest if post-quality does not yet exist
if not os.path.exists(manifest_path):
    manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_metadata.csv")
    print(f"Post-quality manifest not found. Using: {manifest_path}")

d_splits = os.path.join(OUTPUT_ROOT, "splits")
os.makedirs(d_splits, exist_ok=True)
print(f"Source manifest : {manifest_path}")

Source manifest : C:\SKIN CANCER v2\pipe output\manifests\training_eligible_manifest_post_quality.csv


## Build Grouping Policies

In [8]:
df = pd.read_csv(manifest_path)
print(f"Loaded rows: {len(df):,}")
class_counts = df["final_authoritative_label"].value_counts()
print("Class distribution:")
print(class_counts.to_string())

def determine_grouping(row):
    if pd.notna(row.get("lesion_id")):
        return str(row["lesion_id"]), "lesion_id"
    if pd.notna(row.get("family_id")):
        return str(row["family_id"]), "family_id"
    if pd.notna(row.get("canonical_match_id")):
        return str(row["canonical_match_id"]), "canonical_match_id_fallback"
    return str(row["base_id_candidate"]), "base_id_candidate_fallback"

grouping = df.apply(determine_grouping, axis=1)
df["effective_split_group_id"] = [g[0] for g in grouping]
df["grouping_source"]          = [g[1] for g in grouping]

print("\n=== GROUPING SOURCE DISTRIBUTION ===")
display(
    df["grouping_source"].value_counts()
    .rename_axis("grouping_source").reset_index(name="row_count")
)

group_df = (
    df.groupby("effective_split_group_id")
    .agg(
        final_authoritative_label=("final_authoritative_label", lambda x: x.mode().iloc[0]),
        grouping_source=("grouping_source", lambda x: x.mode().iloc[0]),
        row_count=("effective_split_group_id", "size"),
    )
    .reset_index()
)

group_label_counts = (
    df.groupby("effective_split_group_id")["final_authoritative_label"].nunique()
)
multi_class_groups = group_label_counts[group_label_counts > 1]
print(f"\nUnique effective groups       : {len(group_df):,}")
print(f"Groups with >1 class label    : {len(multi_class_groups)}")

if len(multi_class_groups) > 0:
    raise ValueError(
        f"{len(multi_class_groups)} effective groups contain multiple class labels. "
        "Resolve before splitting."
    )

print("\n=== GROUP LABEL DISTRIBUTION ===")
display(
    group_df["final_authoritative_label"].value_counts()
    .rename_axis("group_label").reset_index(name="group_count")
)

Loaded rows: 20,389
Class distribution:
final_authoritative_label
NV     12708
MEL     4430
BCC     3251

=== GROUPING SOURCE DISTRIBUTION ===


,grouping_source,row_count
0,lesion_id,18656
1,family_id,1733



Unique effective groups       : 11,463
Groups with >1 class label    : 0

=== GROUP LABEL DISTRIBUTION ===


,group_label,group_count
0,NV,8522
1,MEL,1636
2,BCC,1305


## Perform Hierarchical Grouped Split

In [9]:
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

print(f"Split ratios: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")

group_train, group_temp = train_test_split(
    group_df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=group_df["final_authoritative_label"],
    random_state=42
)

val_relative = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
group_val, group_test = train_test_split(
    group_temp,
    test_size=(1.0 - val_relative),
    stratify=group_temp["final_authoritative_label"],
    random_state=42
)

for g, lbl in [(group_train, "train"), (group_val, "val"), (group_test, "test")]:
    g = g.copy()
    g["split_assignment"] = lbl

group_train = group_train.copy(); group_train["split_assignment"] = "train"
group_val   = group_val.copy();   group_val["split_assignment"]   = "val"
group_test  = group_test.copy();  group_test["split_assignment"]  = "test"

group_assignments = pd.concat([group_train, group_val, group_test], axis=0)
df_final = df.merge(
    group_assignments[["effective_split_group_id","split_assignment"]],
    on="effective_split_group_id", how="left"
)

if df_final["split_assignment"].isna().any():
    raise ValueError("Some rows did not receive a split_assignment.")

df_train_split = df_final[df_final["split_assignment"] == "train"].copy()
df_val_split   = df_final[df_final["split_assignment"] == "val"].copy()
df_test_split  = df_final[df_final["split_assignment"] == "test"].copy()

print(f"\nTrain rows: {len(df_train_split):,}")
print(f"Val rows  : {len(df_val_split):,}")
print(f"Test rows : {len(df_test_split):,}")

Split ratios: train=0.7, val=0.15, test=0.15

Train rows: 14,332
Val rows  : 3,012
Test rows : 3,045


## Validation & Leakage Checks

In [10]:
print("=== SPLIT SUMMARY ===")
print(f"1. Split Row Counts: Train={len(df_train_split):,} | Val={len(df_val_split):,} | Test={len(df_test_split):,}")

split_class_counts = pd.crosstab(df_final["split_assignment"], df_final["final_authoritative_label"])
print("\n2. Class Counts per Split:")
display(split_class_counts)

group_splits   = df_final.groupby("effective_split_group_id")["split_assignment"].nunique()
leaked_groups  = group_splits[group_splits > 1]
print(f"\n3. Effective Group Leakage Check:")
print(f"   Groups spanning multiple splits: {len(leaked_groups)}")

lesion_rows = df_final[df_final["lesion_id"].notna()].copy() if "lesion_id" in df_final.columns else pd.DataFrame()
leaked_lesions = pd.Series([], dtype=int)
if not lesion_rows.empty:
    lesion_split_counts = lesion_rows.groupby("lesion_id")["split_assignment"].nunique()
    leaked_lesions      = lesion_split_counts[lesion_split_counts > 1]
print(f"\n4. Lesion ID Leakage Check:")
print(f"   Lesion IDs spanning multiple splits: {len(leaked_lesions)}")

sum_totals = len(df_final) == len(df)
print(f"\n5. Total Integrity Check:")
print(f"   Split totals match full dataset: {sum_totals} ({len(df_final):,} rows)")

lesion_coverage = (
    df_final.groupby("split_assignment")
    .agg(
        total_rows=("split_assignment","size"),
        rows_with_lesion_id=("lesion_id", lambda x: x.notna().sum()) if "lesion_id" in df_final.columns else ("split_assignment","size"),
    )
    .reset_index()
)
print("\n6. Lesion ID Coverage by Split:")
display(lesion_coverage)

grouping_source_by_split = pd.crosstab(df_final["split_assignment"], df_final["grouping_source"])
print("\n7. Grouping Source by Split:")
display(grouping_source_by_split)

if len(leaked_groups) > 0:
    raise ValueError(f"{len(leaked_groups)} effective groups leaking across splits.")
if len(leaked_lesions) > 0:
    raise ValueError(f"{len(leaked_lesions)} lesion_id values leaking across splits.")
if not sum_totals:
    raise ValueError("Split totals do not sum back to full dataset.")

print("\nAll integrity checks passed.")

=== SPLIT SUMMARY ===
1. Split Row Counts: Train=14,332 | Val=3,012 | Test=3,045

2. Class Counts per Split:


final_authoritative_label,BCC,MEL,NV
split_assignment,,,
test,512,639,1894
train,2260,3144,8928
val,479,647,1886



3. Effective Group Leakage Check:
   Groups spanning multiple splits: 0

4. Lesion ID Leakage Check:
   Lesion IDs spanning multiple splits: 0

5. Total Integrity Check:
   Split totals match full dataset: True (20,389 rows)

6. Lesion ID Coverage by Split:


,split_assignment,total_rows,rows_with_lesion_id
0,test,3045,2788
1,train,14332,13107
2,val,3012,2761



7. Grouping Source by Split:


grouping_source,family_id,lesion_id
split_assignment,,
test,257,2788
train,1225,13107
val,251,2761



All integrity checks passed.


## Save Split Manifests & Reports

In [11]:
cols_to_save = [
    "full_path", "final_authoritative_label",
    "canonical_match_id", "lesion_id", "family_id",
    "effective_split_group_id", "grouping_source", "split_assignment"
]
cols_to_save = [c for c in cols_to_save if c in df_final.columns]

df_train_split[cols_to_save].to_csv(os.path.join(d_splits, "train_manifest.csv"), index=False)
df_val_split[cols_to_save].to_csv(os.path.join(d_splits,   "val_manifest.csv"),   index=False)
df_test_split[cols_to_save].to_csv(os.path.join(d_splits,  "test_manifest.csv"),  index=False)

pd.DataFrame([
    {"split":"train","total_rows":len(df_train_split)},
    {"split":"val",  "total_rows":len(df_val_split)},
    {"split":"test", "total_rows":len(df_test_split)}
]).to_csv(os.path.join(d_splits,"split_summary_report.csv"), index=False)

grouping_source_by_split.to_csv(os.path.join(d_splits,"split_grouping_report.csv"))
split_class_counts.to_csv(os.path.join(d_splits,"class_distribution_by_split.csv"))

pd.DataFrame([{
    "leaked_effective_groups": len(leaked_groups),
    "leaked_lesion_ids":       len(leaked_lesions),
    "sum_matches_total":       sum_totals
}]).to_csv(os.path.join(d_splits,"split_integrity_checks.csv"), index=False)

print("Split manifests and reports saved.")
print("Downstream stages must use:")
print("  - train_manifest.csv")
print("  - val_manifest.csv")
print("  - test_manifest.csv")

Split manifests and reports saved.
Downstream stages must use:
  - train_manifest.csv
  - val_manifest.csv
  - test_manifest.csv


In [12]:
# Final summary
_train_f = os.path.join(d_splits, "train_manifest.csv")
_val_f   = os.path.join(d_splits, "val_manifest.csv")
_test_f  = os.path.join(d_splits, "test_manifest.csv")

print("=" * 60)
print("  07_split_design -- FINAL SUMMARY")
print("=" * 60)
print(f"\nSource manifest used  : {os.path.basename(manifest_path)}")
print(f"Total rows split      : {len(df_final):,}")
print(f"\nSplit row counts:")
print(f"  Train : {len(df_train_split):,}")
print(f"  Val   : {len(df_val_split):,}")
print(f"  Test  : {len(df_test_split):,}")
print(f"\nClass counts by split:")
for split in ["train","val","test"]:
    sub  = df_final[df_final["split_assignment"] == split]
    ccts = sub["final_authoritative_label"].value_counts()
    print(f"  {split:<5}: NV={int(ccts.get('NV',0)):,}  MEL={int(ccts.get('MEL',0)):,}  BCC={int(ccts.get('BCC',0)):,}")
print(f"\nEffective group leakage count : {len(leaked_groups)}")
print(f"Lesion_id leakage count       : {len(leaked_lesions)}")
print(f"\nOutput file verification:")
for p in [_train_f, _val_f, _test_f,
          os.path.join(d_splits,"split_summary_report.csv"),
          os.path.join(d_splits,"split_integrity_checks.csv")]:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  07_split_design -- FINAL SUMMARY

Source manifest used  : training_eligible_manifest_post_quality.csv
Total rows split      : 20,389

Split row counts:
  Train : 14,332
  Val   : 3,012
  Test  : 3,045

Class counts by split:
  train: NV=8,928  MEL=3,144  BCC=2,260
  val  : NV=1,886  MEL=647  BCC=479
  test : NV=1,894  MEL=639  BCC=512

Effective group leakage count : 0
Lesion_id leakage count       : 0

Output file verification:
  [OK] train_manifest.csv                             1,678,632 bytes
  [OK] val_manifest.csv                                 346,922 bytes
  [OK] test_manifest.csv                                354,128 bytes
  [OK] split_summary_report.csv                              52 bytes
  [OK] split_integrity_checks.csv                            71 bytes
